In [1]:
# @title 1. Mount Google Drive and Set Up File Paths
from google.colab import drive
import os
from pathlib import Path

# Mount your Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Google Drive mounted.")

Mounting Google Drive...
Mounted at /content/drive
Google Drive mounted.


In [2]:
# @title 2. File Paths Set Up

OTHER_PREFIX = ""

# @markdown if u have prefix want to add other the given denoise_prefix, check the bool and the type sth in the OTHER_PREFIX
OTHER_PREFIX_bool = True # @param{type: "boolean"}

if OTHER_PREFIX_bool == True:
    OTHER_PREFIX = "cuda_float" # @param{type: "string"}
    OTHER_PREFIX =  OTHER_PREFIX + "_"
# @markdown ---

EMPIAR_ID = 10017 # @param{type : 'string'}

DATA_DIR_PATH = '/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data'     # @param {type: "string"}
DATA_DIR_PATH = DATA_DIR_PATH + f'/EMPIAR-{EMPIAR_ID}/micrographs'
DATA_DIR = Path(DATA_DIR_PATH)

OUTPUT_DIR_PATH ='/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data'  # @param {type: "string"}
OUTPUT_DIR_PATH = OUTPUT_DIR_PATH + f'/EMPIAR-{EMPIAR_ID}/C-CSN-D_{OTHER_PREFIX}micrographs'
OUTPUT_DIR = Path(OUTPUT_DIR_PATH)

TEMP_DIR_PATH = '/content/temp_denoise'                           # @param {type: "string"}
TEMP_DIR = Path(TEMP_DIR_PATH)


!mkdir {OUTPUT_DIR_PATH} -p
!mkdir {TEMP_DIR_PATH} -p


print(f"\nOriginal Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Temporary Directory: {TEMP_DIR}")

# Create output and temporary directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)


Original Data Directory: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017/micrographs
Output Directory: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017/C-CSN-D_cuda_float_micrographs
Temporary Directory: /content/temp_denoise


In [3]:
# @title 3-0. install mrcfile package
!pip install mrcfile -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.3 MB/s eta 0:00:00


In [4]:
# @title 3. Import Packages
import subprocess
import shutil
import time
from dataclasses import dataclass, field
import sys

import numpy as np
import mrcfile
import cv2
# from numpy.fft import fft2, ifft2

from cupy.fft import fft2, ifft2
from scipy.signal.windows import gaussian
from skimage.exposure import equalize_adapthist
from skimage.restoration import denoise_nl_means

import cupy as cp

In [5]:
# @title 4. Load in Denoise Process Functions Copy From CryoSegNet

# Code for implementing different different image processing techniques for denoising.

# Function to move an array to the GPU
def to_gpu(image):
    return cp.asarray(image)

# Function to move an array back to the CPU
def to_cpu(image):
    return cp.asnumpy(image)

def transform(image):
    i_min = cp.min(image)
    i_max = cp.max(image)
    image = (image - i_min) / (i_max - i_min)
    return image

def standard_scaler(image):
    # Move array to GPU
    image = to_gpu(image.astype(cp.float32))
    kernel_size = 9

    # Gaussian Blur (OpenCV doesn't have a cupy equivalent, so this is a CPU bottleneck)
    image_cpu = to_cpu(image)
    image_cpu = cv2.GaussianBlur(image_cpu, (kernel_size, kernel_size), 0)
    image = to_gpu(image_cpu)

    mu = cp.mean(image)
    sigma = cp.std(image)
    image = (image - mu) / sigma
    image = transform(image)
    return image

def contrast_enhancement(image):
    # This scikit-image function is a CPU bottleneck
    image_cpu = to_cpu(image)
    denoised_image = denoise_nl_means(image_cpu, h=0.1, fast_mode=True, patch_size=5, patch_distance=6)
    return to_gpu(denoised_image)

def gaussian_kernel(kernel_size=3):
    h = gaussian(kernel_size, kernel_size / 3).reshape(kernel_size, 1)
    h = cp.asarray(h) # Move kernel to GPU
    h = cp.dot(h, h.transpose())
    h /= cp.sum(h)
    return h.astype(cp.float32)

def wiener_filter(img, kernel, K):
    # Ensure all operations are using cupy
    dummy = cp.copy(img)
    # Use cupy's fft2
    dummy = fft2(dummy)
    # Use cupy's fft2
    kernel = fft2(kernel, s=img.shape)
    kernel = cp.conj(kernel) / (cp.abs(kernel) ** 2 + K)
    dummy = dummy * kernel
    dummy = cp.abs(ifft2(dummy))
    return dummy.astype(cp.float32)

def clahe(image):
    # This scikit-image function is a CPU bottleneck
    image_cpu = to_cpu(image)
    img_equalized = equalize_adapthist(image_cpu, clip_limit=0.03)
    return to_gpu(img_equalized.astype(cp.float32))

def guided_filter(input_image, guidance_image, radius=20, epsilon=0.1):
    # OpenCV's boxFilter is a CPU bottleneck
    input_image_cpu = to_cpu(input_image)
    guidance_image_cpu = to_cpu(guidance_image)

    mean_guidance_cpu = cv2.boxFilter(guidance_image_cpu, -1, (radius, radius))
    mean_input_cpu = cv2.boxFilter(input_image_cpu, -1, (radius, radius))

    mean_guidance = to_gpu(mean_guidance_cpu)
    mean_input = to_gpu(mean_input_cpu)

    mean_guidance_input_cpu = cv2.boxFilter(guidance_image_cpu * input_image_cpu, -1, (radius, radius))
    mean_guidance_input = to_gpu(mean_guidance_input_cpu)

    covariance_guidance_input = mean_guidance_input - mean_guidance * mean_input

    mean_guidance_sq_cpu = cv2.boxFilter(guidance_image_cpu * guidance_image_cpu, -1, (radius, radius))
    mean_guidance_sq = to_gpu(mean_guidance_sq_cpu)

    variance_guidance = mean_guidance_sq - mean_guidance * mean_guidance

    a = covariance_guidance_input / (variance_guidance + epsilon)
    b = mean_input - a * mean_guidance

    mean_a_cpu = cv2.boxFilter(to_cpu(a), -1, (radius, radius))
    mean_b_cpu = cv2.boxFilter(to_cpu(b), -1, (radius, radius))

    mean_a = to_gpu(mean_a_cpu)
    mean_b = to_gpu(mean_b_cpu)

    output_image = mean_a * guidance_image + mean_b

    return output_image.astype(cp.float32)

Gaussian_kernel_size = 9
kernel = gaussian_kernel(Gaussian_kernel_size)

def denoise(image_path):
    import mrcfile
    image = mrcfile.read(image_path)

    normalized_image = standard_scaler(np.array(image))

    contrast_enhanced_image = contrast_enhancement(normalized_image)
    weiner_filtered_image = wiener_filter(contrast_enhanced_image, kernel, K=30)
    clahe_image = clahe(weiner_filtered_image)
    guided_filter_image = guided_filter(clahe_image, weiner_filtered_image)

    return to_cpu(guided_filter_image)

In [ ]:
# @title 5. Denoise the mrcfile

paths = DATA_DIR.rglob('*.mrc')
paths = sorted(list(paths))
n = 0
for path in paths:
    n += 1
    denoised_img = denoise(path)
    mrcfile.write(OUTPUT_DIR / path.name, denoised_img, overwrite=True)
    print(f"{n}. Denoised image saved to {OUTPUT_DIR / path.name}")

1. Denoised image saved to /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/data/EMPIAR-10017/C-CSN-D_cuda_float_micrographs/Falcon_2012_06_12-14_33_35_0.mrc


---
### Result visualization

In [ ]:
# @title load in function for image reading
import os

# You may need to install mrcfile with: !pip install mrcfile

def load_mrc(path: str, standardize: bool = False):
    with mrcfile.open(path) as f:
        image = f.data
        header = f.header
        extended_header = f.extended_header

    if image.dtype == np.float16:
        image = image.astype(np.float32)

    if standardize:
        # Standardize based on header info if available, otherwise on mean/std
        if hasattr(header, 'amean') and hasattr(header, 'rms'):
            image = image - header.amean
            image /= header.rms
        else:
            image = (image - image.mean()) / image.std()

    return image, header, extended_header


def load_image(path: str, standardize: bool = False, make_image: bool = True):
    ext = os.path.splitext(path)[1]

    if ext == '.mrc':
        data = load_mrc(path, standardize)
    else:
        # Placeholder for other file types, though they are removed for brevity.
        raise ValueError("Unsupported file type.")

    (image, header, extended_header) = data if isinstance(data, tuple) else (data, None, None)

    return (image, header, extended_header) if header else image


In [ ]:
import matplotlib.pyplot as plt

name = 'Falcon_2012_06_12-14_33_35_0' # @param {type: "string"}

# Load the raw micrograph, ensuring it's a NumPy array.
mic_raw = load_image(os.path.join(DATA_DIR_PATH, f"{name}.mrc"), standardize=True)[0]


# Load the denoised micrograph, ensuring it's a NumPy array.
mic_dn = load_image(os.path.join(OUTPUT_DIR_PATH, f"{name}.mrc"), standardize=True)[0]


# Plotting the images
_, ax = plt.subplots(1, 2, figsize=(24, 12))

ax[0].imshow(mic_raw, vmin=-4, vmax=4, cmap='Greys_r')
ax[1].imshow(mic_dn, vmin=-4, vmax=4, cmap='Greys_r')

plt.show()